# Create FFG Awards (Oesterreichische Forschungsfoerderungsgesellschaft)

Creates FFG (Austrian Research Promotion Agency) awards from the agency's own
public **FFG Projektdatenbank** at https://projekte.ffg.at/ ("Kurzinformationen
zu laufenden und abgeschlossenen Projekten, die ueber die FFG eine Foerderung
erhalten haben"; focus on projects from 2015; published only with the funding
recipient's consent; FFG's small formats such as feasibility/innovation
checks are not included). **8,164 projects, start years 2013-2027, 100% title/dates/lead organisation, 53% abstract, no amounts or person names** (local run 2026-09-30).

**Source = the database's own Excel export** (runbook ladder item 0): the
search list's "als Excel exportieren" POSTs selected project ids to
`/projekt/excel`. The script pages the full unfiltered search list to collect
every Projekt-ID, exports them in batches of 100, and collapses the
(project x organisation) rows to one row per project. data.gv.at /
data.europa.eu carry no FFG project dataset (checked 2026-09-30).

**Prerequisites:**
- Run `scripts/local/ffg_to_s3.py` first. It uploads
  `s3://openalex-ingest/awards/ffg/ffg_projects.parquet` (§1.4 shrink guard).

**`funder_award_id` (§2.1.1) -- read this before Step 7.** The database and its
export publish **no FFG project number**. Their only identifier is
`Projekt-ID`, a GUID (e.g. `ff784a3e-f36b-1410-85fe-00d3c07d99bb`) that is also
the public project URL. Citing works quote FFG's 6-7 digit project numbers
(e.g. `872176`, `FFG-892418`, `FO999902549`; citation-side sample from
crossref_work.grants / crossref_work_funders for F4320323031, 2026-09-30,
~5,600 stub rows), which cannot be derived from the GUID. So these awards
**will not collapse onto the existing citation stubs**; some FFG grants will
exist twice (this record + a bare stub) until FFG publishes project numbers.
We ship the GUID because it is the funder's own published Projekt-ID; the
coordinator should weigh this split-brain cost before integrating (Step 7).

**No amounts and no person names** are published. `lead_investigator` carries
only the lead organisation (Konsortialfuehrer / Einzelantragsteller) as
`affiliation.name`, with given/family NULL; Step 6.7 amount check waived.

**Funder details (Path A, F4320* Crossref-registered):**
- funder_id: `4320323031`
- display_name: Oesterreichische Forschungsfoerderungsgesellschaft (FFG)
- ror_id / doi: from `openalex.funders.funders` (ror 028jc0449, doi 10.13039/501100004955)

**Priority:** `489` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.ffg_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/ffg/ffg_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_projects FROM openalex.awards.ffg_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.ffg_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.ffg_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320323031 is a Crossref-registered funder, so it MUST resolve to exactly 1
row in `openalex.funders.funders`. If 0 rows, STOP (do not proceed) and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320323031;

## Step 2: Create FFG Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.ffg_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320323031  -- FFG
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.project_id))))) % 9000000000 as id,
    COALESCE(NULLIF(TRIM(g.title), ''), NULLIF(TRIM(g.acronym), '')) as display_name,
    NULLIF(TRIM(g.abstract), '') as description,
    f.funder_id,
    LOWER(TRIM(g.project_id)) as funder_award_id,
    CAST(NULL AS DOUBLE) as amount,
    CAST(NULL AS STRING) as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    'research' as funding_type,
    NULLIF(TRIM(g.programme), '') as funder_scheme,
    'ffg_projektdatenbank' as provenance,
    TRY_TO_DATE(g.start_date, 'yyyy-MM-dd') as start_date,
    TRY_TO_DATE(g.end_date, 'yyyy-MM-dd') as end_date,
    YEAR(TRY_TO_DATE(g.start_date, 'yyyy-MM-dd')) as start_year,
    YEAR(TRY_TO_DATE(g.end_date, 'yyyy-MM-dd')) as end_year,
    -- Lead organisation only (no person names are published).
    CASE WHEN NULLIF(TRIM(g.lead_org), '') IS NOT NULL THEN named_struct(
        'given_name', CAST(NULL AS STRING),
        'family_name', CAST(NULL AS STRING),
        'orcid', CAST(NULL AS STRING),
        'role_start', CAST(NULL AS DATE),
        'affiliation', named_struct(
            'name', TRIM(g.lead_org),
            'country', CASE UPPER(TRIM(g.lead_org_country))
                WHEN 'AUT' THEN 'AT' WHEN 'DEU' THEN 'DE' WHEN 'CHE' THEN 'CH' WHEN 'NLD' THEN 'NL'
                WHEN 'GBR' THEN 'GB' WHEN 'ITA' THEN 'IT' WHEN 'FRA' THEN 'FR' WHEN 'BEL' THEN 'BE'
                WHEN 'SWE' THEN 'SE' WHEN 'ESP' THEN 'ES' WHEN 'CZE' THEN 'CZ' WHEN 'SVN' THEN 'SI'
                WHEN 'HUN' THEN 'HU' WHEN 'SVK' THEN 'SK' WHEN 'POL' THEN 'PL' WHEN 'DNK' THEN 'DK'
                WHEN 'FIN' THEN 'FI' WHEN 'NOR' THEN 'NO' WHEN 'USA' THEN 'US' WHEN 'LUX' THEN 'LU'
            END,
            'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
        )
    ) END as lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
         affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>)
        as co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
         affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>>)
        as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.project_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM openalex.awards.ffg_raw g
CROSS JOIN src_funder f
WHERE g.project_id IS NOT NULL AND TRIM(g.project_id) != '';

In [ ]:
%sql
-- Shape check (§2.1.1): funder_award_id is the Projekt-ID GUID, unique.
SELECT
    COUNT(*) AS total,
    COUNT(DISTINCT id) AS distinct_ids,
    SUM(CASE WHEN funder_award_id RLIKE '^[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}$' THEN 0 ELSE 1 END) AS bad_shape
FROM openalex.awards.ffg_awards;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'ffg_projektdatenbank' AND priority = 489;

-- Insert into openalex_awards_raw with priority.
-- Priority 489: direct-from-funder ingest of the FFG Projektdatenbank Excel export.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    489 as priority
FROM openalex.awards.ffg_awards;

## Verification Queries

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, start_date, end_date,
       lead_investigator.affiliation.name, lead_investigator.affiliation.country
FROM openalex.awards.ffg_awards LIMIT 20;

In [ ]:
%sql
SELECT funder_scheme, COUNT(*) as cnt
FROM openalex.awards.ffg_awards
GROUP BY funder_scheme ORDER BY cnt DESC LIMIT 50;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt
FROM openalex.awards.ffg_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- Section 6.4a frequency check: lead organisation + titles must be a real long-tail.
SELECT lead_investigator.affiliation.name AS lead_org, COUNT(*) AS n
FROM openalex.awards.ffg_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.ffg_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.3 coverage (amount waived: not published).
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(start_date) as has_start_date,
    COUNT(end_date) as has_end_date,
    COUNT(lead_investigator.affiliation.name) as has_lead_org,
    COUNT(amount) as has_amount
FROM openalex.awards.ffg_awards;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'ffg_projektdatenbank'
GROUP BY provenance, priority;